# RAG based Document Question Answering System

**Aim:** Build a simple Retrieval-Augmented Generation (RAG) pipeline that can answer questions from a custom document.

**Steps:**
1. Load a document (PDF/TXT)
2. Split it into chunks
3. Convert chunks into embeddings
4. Store embeddings in a vector database (FAISS)
5. Retrieve relevant chunks for a given question
6. Use an LLM to generate the final answer


## 1. Install dependencies

In [ ]:
# !pip install -q -U langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers transformers pypdf

^C


## 2. Import libraries

In [2]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from transformers import pipeline

c:\Users\harsh\Downloads\RAG_Document_QA\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load the document
Upload your own PDF/TXT file to Colab and change the path below.

In [3]:
doc_path = "sample_doc.txt"   # change this to your file

loader = PyPDFLoader(doc_path) if doc_path.endswith(".pdf") else TextLoader(doc_path)
documents = loader.load()
print(f"Loaded {len(documents)} document section(s)")

Loaded 1 document section(s)


## 4. Split document into chunks

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 4 chunks


## 5. Create embeddings and store in FAISS

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1825.83it/s]


## 6. Load LLM and build the RAG chain

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_answer(prompt, max_length=256):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_length=max_length)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1443.90it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 7. Ask questions

In [14]:
def ask(question):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
    answer = generate_answer(prompt)
    print("Q:", question)
    print("A:", answer)
    print("\nRetrieved chunks used:")
    for i, doc in enumerate(docs, 1):
        print(f"{i}. {doc.page_content[:150]}...")
    print("-"*80)

In [15]:
ask("What is RAG?")

Q: What is RAG?
A: RAG is a technique that combines information retrieval with text generation.

Retrieved chunks used:
1. Retrieval-Augmented Generation (RAG) Overview

RAG is a technique that combines information retrieval with text generation.
Instead of relying only on...
2. Benefits of RAG:
- Reduces hallucination by grounding answers in real source text.
- Works with private or domain-specific data without retraining the...
3. How a RAG pipeline works:
1. Load documents (PDF, TXT, DOCX, etc.)
2. Split the documents into smaller chunks (e.g. 500 characters) so they fit
   wit...
--------------------------------------------------------------------------------


In [18]:
ask("Why does RAG reduce hallucination?")
ask("What embedding model is used in this pipeline?")

Q: Why does RAG reduce hallucination?
A: Reduces hallucination by grounding answers in real source text

Retrieved chunks used:
1. Benefits of RAG:
- Reduces hallucination by grounding answers in real source text.
- Works with private or domain-specific data without retraining the...
2. Retrieval-Augmented Generation (RAG) Overview

RAG is a technique that combines information retrieval with text generation.
Instead of relying only on...
3. How a RAG pipeline works:
1. Load documents (PDF, TXT, DOCX, etc.)
2. Split the documents into smaller chunks (e.g. 500 characters) so they fit
   wit...
--------------------------------------------------------------------------------
Q: What embedding model is used in this pipeline?
A: all-MiniLM-L6-v2

Retrieved chunks used:
1. How a RAG pipeline works:
1. Load documents (PDF, TXT, DOCX, etc.)
2. Split the documents into smaller chunks (e.g. 500 characters) so they fit
   wit...
2. database for the most similar chunks (top-k retrieval).
6. Pass th

In [16]:
ask("What are the benefits of using RAG?")

Q: What are the benefits of using RAG?
A: Works with private or domain-specific data without retraining the model.

Retrieved chunks used:
1. Benefits of RAG:
- Reduces hallucination by grounding answers in real source text.
- Works with private or domain-specific data without retraining the...
2. Retrieval-Augmented Generation (RAG) Overview

RAG is a technique that combines information retrieval with text generation.
Instead of relying only on...
3. How a RAG pipeline works:
1. Load documents (PDF, TXT, DOCX, etc.)
2. Split the documents into smaller chunks (e.g. 500 characters) so they fit
   wit...
--------------------------------------------------------------------------------


In [17]:
ask("Which tools are commonly used to build a RAG system?")

Q: Which tools are commonly used to build a RAG system?
A: LangChain, LlamaIndex, FAISS, Chroma, Hugging Face Transformers, and Sentence-Transformers

Retrieved chunks used:
1. Benefits of RAG:
- Reduces hallucination by grounding answers in real source text.
- Works with private or domain-specific data without retraining the...
2. Retrieval-Augmented Generation (RAG) Overview

RAG is a technique that combines information retrieval with text generation.
Instead of relying only on...
3. How a RAG pipeline works:
1. Load documents (PDF, TXT, DOCX, etc.)
2. Split the documents into smaller chunks (e.g. 500 characters) so they fit
   wit...
--------------------------------------------------------------------------------


## Conclusion

This notebook implements a basic RAG pipeline:
- Documents are split into chunks and embedded using `all-MiniLM-L6-v2`
- Chunks are stored and searched using FAISS (similarity search, top-3)
- Retrieved chunks are passed to `flan-t5-base` to generate a grounded answer

This avoids hallucination since the model only answers using the retrieved context, and it works on any custom document (notes/resume/research paper/etc.) without retraining the model.